# Reproducing the interpretability results (Section 4.3)

This notebook reproduces the interpretability analysis of the paper with the Gaussian
mixture of $K = 7$ components used there:

- **Fig. 2**, the physics profile of each mixture component;
- the **localisation figures**, showing where the flagged signal falls among the components;
- the **top-observable figure**, the observable ranked first, flagged signal against local SM;
- **Table 6**, the Wasserstein ranking of the observables within each component;
- the **Spearman figure**, the correlation between the autoencoder score and each observable.

It runs the repository's scripts in the order used for the paper. The notebook adds no
analysis code of its own: it makes the calls, then checks every number against the paper.
The whole notebook runs in about 15 minutes on a CPU.

### Inputs

- the **saved embeddings** of the VCReg encoder ($d_\text{model} = 256$, seed 3) for the
  12 SM classes and the 3 proxy signals;
- the **saved mixture**, $K = 7$, fitted on those embeddings;
- the encoder checkpoint and its QCD-only autoencoder (README stage 3), with the threshold
  calibrated on validation;
- the SM-normalised test datasets, and the CASE production (README stage 1) for
  $Z' \to n(\mu\mu)$.

The embeddings and the mixture are loaded rather than recomputed. A new extraction draws
a different training sample, because the data configuration sets no seed. A new fit of
the mixture on the saved embeddings reproduces it inside the repository's container
(`fm_testing.sif`, where the paper's fit ran) to $10^{-4}$; with other library versions
it converges to essentially the same partition but numbers its components differently,
so the checks below, written for the paper's components, would not apply. README stage 4
lists the commands that produced both.

## Configuration

The only cell to edit. The inputs are published on Zenodo: `bash
scripts/download_xai_data.sh` downloads them (about 4 GB), checks them and unpacks them
into `data/`, where this cell finds them. Without `data/`, it reads the same files from
their original location on CERN EOS. `OUT` receives everything the notebook writes;
`REF` is the directory of the published outputs, read only by the final comparison, so
`OUT` must be a different directory.

In [ ]:
import csv, json, os, shutil, subprocess, sys, time
from pathlib import Path

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
PY = sys.executable

# Inputs from Zenodo: `bash scripts/download_xai_data.sh` unpacks them into data/.
DATA = Path(os.environ.get("XAI_DATA", REPO / "data"))
REF  = Path("/eos/user/d/dgenoves/anomaly_pipeline/xai_paper")   # published outputs (optional)

if (DATA / "embeddings").exists():
    EMBEDDINGS = DATA / "embeddings"                           # saved embeddings
    MIXTURE    = DATA / "mixture/gmm_K7.pkl"                   # saved mixture
    ENC        = DATA / "checkpoints/encoder/epoch_014.ckpt"
    AE         = DATA / "checkpoints/autoencoder/ae-epochepoch=49.ckpt"
    VEC_TEST   = DATA / "test/vectorized"                      # observables
    PRE_TEST   = DATA / "test/preprocessed"                    # encoder input
    CASE_DATA  = DATA / "case"
    CASE_SRC   = DATA / "case_src"         # CASE parquet, for the untruncated lepton count
else:
    # the same files at their original location on CERN EOS
    NE  = Path("/eos/user/d/dgenoves/anomaly_pipeline/new_exp")
    FMD = Path("/eos/user/d/dgenoves/foundation_model_testing_data")
    RUN = "vcreg_12class_nosparse_dmodel256_cern"
    EMBEDDINGS = NE / "xai_embeddings_smnorm" / RUN / "encoder_seed_3/embeddings"
    MIXTURE    = REF / "k_selection_v3/vcreg_d256_seed3_diag_pca64/gmm_K7.pkl"
    ENC        = NE / RUN / "seed_3/checkpoints/epoch_014.ckpt"
    AE         = NE / "ad_results" / RUN / "encoder_seed_3/mse_normal/checkpoints/ae-epochepoch=49.ckpt"
    VEC_TEST   = FMD / "v2_nosparse_higgs_allsm_highlevel/vectorized/test"
    PRE_TEST   = FMD / "v2_nosparse_higgs_smnorm_highlevel/preprocessed/test"
    CASE_DATA  = FMD / "v2_nosparse_case_smnorm_highlevel"
    CASE_SRC   = FMD / "_case_src"

OUT = Path(os.environ.get("XAI_OUT", REPO / "outputs" / "xai_reproduce"))
FORCE = False        # True reruns steps whose output already exists

assert OUT.resolve() != REF.resolve(), "OUT must not be the published output tree"
for p in (EMBEDDINGS, MIXTURE, ENC, AE, VEC_TEST, PRE_TEST, CASE_DATA):
    assert p.exists(), f"missing input: {p}"
OUT.mkdir(parents=True, exist_ok=True)
print("repository:", REPO)
print("inputs    :", DATA if (DATA / "embeddings").exists() else "original EOS locations")
print("outputs   :", OUT)

Output paths and a helper. The tree under `OUT` has the same layout as the published
one, so the paper's figure scripts can read it through their `--xp` option. `run()`
calls one script and skips it when its output already exists.

In [ ]:
MH   = OUT / "vcreg_d256_seed3_smnorm/04_profile/matched_sm_hh4b.npz"
MV   = OUT / "case_HVdilep_Zp1000_piD2_mumu_d256_seed3/matched_sm_HVdilep_Zp1000_piD2_mumu.npz"
RANK = OUT / "rank_k7_sm"
FIG  = OUT / "figures"
PCA  = ["--pca-dim", 64, "--pca-embeddings-dir", EMBEDDINGS, "--pca-seed", 3]
SIGNALS = {"hh4b": (13, MH), "hvdilep": (20, MV)}   # name: (label, event array)
QUIET = ("warn", "had enough events")   # log lines not echoed: warnings, per-class counts


def run(script, *args, done=None):
    """Run one repository script from the repo root; skip it if `done` already exists."""
    if done is not None and Path(done).exists() and not FORCE:
        print(f"skip  {script}: {Path(done).relative_to(OUT)} exists")
        return
    cmd = [PY, "-u", script, *map(str, args)]
    print("$", " ".join(cmd), flush=True)
    t0 = time.time()
    # No progress bars: read through a pipe, every refresh would print as a new line.
    env = {**os.environ, "PROJECT_ROOT": str(REPO), "PYTHONWARNINGS": "ignore",
           "TQDM_DISABLE": "1"}
    proc = subprocess.Popen(cmd, cwd=REPO, env=env, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        if not any(s in line.lower() for s in QUIET):
            print(line, end="")
    if proc.wait():
        raise RuntimeError(f"{script} failed with exit code {proc.returncode}")
    print(f"      done in {(time.time() - t0) / 60:.1f} min\n")

## 1. Saved embeddings and mixture

The mixture has diagonal covariance and lives in the space of the first 64 principal
components of the SM training embeddings. That projection is an exact SVD, so each step
rebuilds it from `EMBEDDINGS` instead of storing it; for the same reason the mixture
must be used with the embeddings it was fitted on.

In [ ]:
import joblib, numpy as np

for split in ("train", "val", "test"):
    with np.load(EMBEDDINGS / f"{split}_embeddings.npz") as d:
        y = d["labels"]
    print(f"{split:5s} embeddings: {len(y):>9,} events, {len(np.unique(y))} classes")

gmm = joblib.load(MIXTURE)
print(f"\nmixture: K = {gmm.n_components}, covariance '{gmm.covariance_type}', "
      f"{gmm.means_.shape[1]} dimensions")
print("component weights:", np.round(gmm.weights_, 3))

## 2. SM + $HH \to 4b$ event array

The population every later step works on: 20,000 test events for each of the twelve SM
classes and for $HH \to 4b$, each with its embedding and its high-level observables
($H_T$, MET, $n_\text{jets}$, $n_{b\text{-tag}}$, $n_\text{leptons}$, $M_{jj}$,
$|\Delta\eta_{jj}|$, $M_T$). The observables come from the vectorised data and the
embeddings from the preprocessed data, matched event by event.

In [ ]:
run("scripts/xai/04_profile_and_rank.py", "--ckpt-path", ENC, "--signal-label", 13,
    "--vectorized-dir", VEC_TEST, "--preproc-split-dir", PRE_TEST,
    "--save-matched", MH, "--output-dir", MH.parent, done=MH)

## 3. SM + $Z' \to n(\mu\mu)$ event array

The same SM events as in step 2, with the $Z' \to n(\mu\mu)$ events of the CASE
production in place of $HH \to 4b$, embedded with the same encoder. The lepton
multiplicity of the signal is read from the original parquet files: the vectorised data
keep at most eight muons, and 38% of these events have more.

In [ ]:
run("scripts/xai/build_matched_case.py", "--case-label", "HVdilep_Zp1000_piD2_mumu",
    "--signal-label", 20, "--sm-matched", MH, "--ckpt", ENC, "--output", MV,
    "--case-data", CASE_DATA, "--case-src", CASE_SRC, done=MV)

## 4. Analysis, for each signal

| Script | What it computes | Used for |
|---|---|---|
| `03_assign_flagged.py` | the component of every event, and whether the autoencoder flags it | localisation figures, top-observable figure, Fig. 2 |
| `04_profile_and_rank.py` | within each component, the Wasserstein-1 distance of each observable between flagged signal and local SM | Table 6, per-component distributions (appendix) |
| `06_ae_mechanism.py` | the Spearman correlation between autoencoder score and each observable, over the flagged signal | Spearman figure |

The autoencoder scores the full 256-dimensional embedding; only the mixture uses the
64-dimensional projection. An event is flagged when its score exceeds the threshold
calibrated on validation at a 10% false-positive rate on QCD.

In [ ]:
for t, (label, M) in SIGNALS.items():
    D = OUT / f"k7_{t}_pca64_d256_seed3"
    run("scripts/xai/03_assign_flagged.py", "--matched-npz", M, "--signal-label", label,
        "--gmm-path", MIXTURE, "--ae-checkpoint", AE, "--output-dir", D / "03_assign_matched",
        "--fpr", 0.10, "--ylim", 0.95, *PCA, done=D / "03_assign_matched/assignments.npz")
    run("scripts/xai/04_profile_and_rank.py", "--matched-npz", M, "--signal-label", label,
        "--gmm-path", MIXTURE, "--ae-checkpoint", AE, "--output-dir", RANK / t,
        "--min-frac", 0.05, "--fpr", 0.10, *PCA, done=RANK / t / "wasserstein_rank.csv")
    run("scripts/xai/06_ae_mechanism.py", "--matched-npz", M, "--gmm-path", MIXTURE,
        "--ae-checkpoint", AE, "--profile-meta", RANK / t / "profile_meta.json",
        "--output-dir", D / "06_ae_mechanism", *PCA,
        done=D / "06_ae_mechanism/ae_mechanism.json")

## 5. Figures and tables

Written to `OUT/figures/`, with the file names used in the paper (`paper/figures/xai/`).
Fig. 2 is shown below.

In [ ]:
FIG.mkdir(parents=True, exist_ok=True)
H, V = OUT / "k7_hh4b_pca64_d256_seed3", OUT / "k7_hvdilep_pca64_d256_seed3"

# Fig. 2 -- the script reads <run-dir>/04_profile/
prof = RANK / "hh4b/04_profile"
prof.mkdir(exist_ok=True)
shutil.copy2(RANK / "hh4b/profile_meta.json", prof / "profile_meta.json")
if not (prof / "matched_sm_hh4b.npz").exists():
    (prof / "matched_sm_hh4b.npz").symlink_to(MH)
for ext in ("pdf", "png"):
    run("scripts/xai/plot_04_profiles.py", "--run-dir", RANK / "hh4b",
        "--assignments", H / "03_assign_matched/assignments.npz",
        "--mark", r"$HH \to 4b$:4,5", "--mark", r"$Z^{\prime} \to n(\mu\mu)$:2",
        "--output", FIG / f"component_profiles_K7.{ext}")

# Spearman figure (only C2 is shown for Z')
run("scripts/xai/plot_06_convergence.py", "--run-dir", H, "--output", FIG / "spearman_hh4b_K7.pdf")
run("scripts/xai/plot_06_convergence.py", "--run-dir", V, "--components", 2,
    "--output", FIG / "spearman_hvdilep_K7.pdf")

# top observable, Table 6
run("paper/figures/make_top_observable.py", "--xp", OUT, "--output", FIG / "top_observable_K7.pdf")
run("paper/make_wasserstein_table.py", "--xp", RANK, "--output", FIG / "wasserstein_side_by_side.tex")

# figures the analysis scripts draw themselves, under the paper's names
for src, name in [
    (H / "03_assign_matched/plots/flagged_assignment.pdf", "localisation_hh4b_K7.pdf"),
    (V / "03_assign_matched/plots/flagged_assignment.pdf", "localisation_hvdilep_K7.pdf"),
    (RANK / "hh4b/plots/hh4b_vs_sm_k5.pdf",              "dist8_hh4b_C5_K7.pdf"),
    (RANK / "hh4b/plots/hh4b_vs_sm_k4.pdf",              "dist8_hh4b_C4_K7.pdf"),
    (RANK / "hvdilep/plots/hv_zp1000_mumu_vs_sm_k2.pdf", "dist8_hvdilep_C2_K7.pdf"),
    (RANK / "hvdilep/plots/hv_zp1000_mumu_vs_sm_k5.pdf", "dist8_hvdilep_C5_K7.pdf"),
    (RANK / "hh4b/plots/physics_per_component.pdf",     "physics_per_component_K7.pdf"),
]:
    shutil.copy2(src, FIG / name)

for f in sorted(FIG.iterdir()):
    print(f"  {f.name:34s} {f.stat().st_size / 1024:7.1f} kB")

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(FIG / "component_profiles_K7.png"), width=820))

## 6. Check against the paper

Each number is compared with the value printed in the paper, rounded to the printed
precision. The paper sources are not tracked by git; in a working copy where they exist,
the regenerated Table 6 is also compared with
`paper/sections/xai/wasserstein_side_by_side.tex`, character by character.

In [ ]:
rows = lambda p: list(csv.DictReader(open(p)))

def w1(t, comp, var, col="W1_sm"):
    r = next(r for r in rows(RANK / t / "wasserstein_rank.csv")
             if int(r["component"]) == comp and r["variable"] == var)
    return float(r[col])

def first(t, comp):
    ranked = [r for r in rows(RANK / t / "wasserstein_rank.csv")
              if int(r["component"]) == comp and r["rank"]]
    return min(ranked, key=lambda r: int(r["rank"]))["variable"]

def frac(t, comp):
    return json.load(open(RANK / t / "profile_meta.json"))["frac_per_component"][comp]

def n_local(t, comp, col):
    return int(next(r for r in rows(RANK / t / "wasserstein_rank.csv")
                    if int(r["component"]) == comp)[col])

def rho(t, comp, var):
    d = json.load(open(OUT / f"k7_{t}_pca64_d256_seed3/06_ae_mechanism/ae_mechanism.json"))
    r = next(r for r in d["results"] if r["component"] == comp)
    return r["spearman_mse_vs_physics"]["flagged"][var]["rho"]

# (where in the paper, quantity, printed value, reproduced value, decimals; None = exact)
CHECKS = [
    ("Table 6",  "HH->4b flagged in C5",          0.556,       frac("hh4b", 5), 3),
    ("Table 6",  "HH->4b flagged in C4",          0.438,       frac("hh4b", 4), 3),
    ("Table 6",  "Z' flagged in C2",              0.88,        frac("hvdilep", 2), 2),
    ("Table 6",  "HH->4b C5: first observable",   "n_bjets",   first("hh4b", 5), None),
    ("Table 6",  "HH->4b C5: W1 vs local SM",     1.73,        w1("hh4b", 5, "n_bjets"), 2),
    ("Table 6",  "HH->4b C5: W1 vs local QCD",    1.98,        w1("hh4b", 5, "n_bjets", "W1_qcd"), 2),
    ("Table 6",  "HH->4b C4: first observable",   "n_bjets",   first("hh4b", 4), None),
    ("Table 6",  "HH->4b C4: W1 vs local SM",     0.92,        w1("hh4b", 4, "n_bjets"), 2),
    ("Table 6",  "HH->4b C4: W1 vs local QCD",    1.37,        w1("hh4b", 4, "n_bjets", "W1_qcd"), 2),
    ("Table 6",  "Z' C2: first observable",       "n_leptons", first("hvdilep", 2), None),
    ("Table 6",  "Z' C2: W1 vs local SM",         4.76,        w1("hvdilep", 2, "n_leptons"), 2),
    ("Table 6",  "Z' C2: QCD events",             1,           n_local("hvdilep", 2, "n_qcd"), None),
    ("Table 6",  "Z' C2: SM events",              29106,       n_local("hvdilep", 2, "n_sm"), None),
    ("Spearman", "HH->4b C5, n_b-tag",            0.43,        rho("hh4b", 5, "n_bjets"), 2),
    ("Spearman", "HH->4b C4, n_b-tag",            0.31,        rho("hh4b", 4, "n_bjets"), 2),
    ("Spearman", "HH->4b C4, n_jets",             0.33,        rho("hh4b", 4, "n_jets"), 2),
    ("Spearman", "Z' C2, H_T",                    0.39,        rho("hvdilep", 2, "HT"), 2),
    ("Spearman", "Z' C2, M_jj",                   0.39,        rho("hvdilep", 2, "Mjj"), 2),
    ("Spearman", "Z' C2, n_leptons",              0.28,        rho("hvdilep", 2, "n_leptons"), 2),
]

n_ok = 0
for where, what, paper, got, dec in CHECKS:
    ok = (got == paper) if dec is None else (round(got, dec) == paper)
    n_ok += ok
    shown = got if dec is None else f"{got:.{dec + 2}f}"
    print(f"{'ok  ' if ok else 'DIFF'} {where:9s} {what:32s} paper {str(paper):>10s}   here {shown}")

# paper/ build products are not tracked by git, so the reference .tex exists only in a
# working copy where the paper has been built.
ref_tex = REPO / "paper/sections/xai/wasserstein_side_by_side.tex"
n_checks = len(CHECKS)
if ref_tex.exists():
    same = (FIG / "wasserstein_side_by_side.tex").read_text() == ref_tex.read_text()
    n_ok, n_checks = n_ok + same, n_checks + 1
    print(f"{'ok  ' if same else 'DIFF'} Table 6   regenerated .tex identical to {ref_tex.relative_to(REPO)}")
else:
    print(f"     Table 6   {ref_tex.relative_to(REPO)} not in this checkout; whole-table comparison skipped")
print(f"\n{n_ok} of {n_checks} match the paper")

### Against the published outputs

When the published outputs are reachable (`REF`), the comparison is exact rather than
at printed precision: the component and the autoencoder flag of every event, and every
distance and rank behind Table 6.

In [ ]:
if not REF.exists():
    print(f"{REF} not reachable; skipping")
else:
    for t in SIGNALS:
        a = np.load(OUT / f"k7_{t}_pca64_d256_seed3/03_assign_matched/assignments.npz")
        b = np.load(REF / f"k7_{t}_pca64_d256_seed3/03_assign_matched/assignments.npz")
        same = (a["assignments"] == b["assignments"]).mean()
        flag = (a["ae_flagged"] == b["ae_flagged"]).mean()
        print(f"{t:8s} {len(a['assignments']):,} events: same component {same:.6f}, "
              f"same autoencoder flag {flag:.6f}")
        new = {(r["component"], r["variable"]): r for r in rows(RANK / t / "wasserstein_rank.csv")}
        old = {(r["component"], r["variable"]): r for r in rows(REF / "rank_k7_sm" / t / "wasserstein_rank.csv")}
        assert new.keys() == old.keys(), "different components or observables ranked"
        diff = max(abs(float(new[k][c]) - float(old[k][c]))
                   for k in old for c in ("W1_sm", "W1_qcd") if old[k][c] not in ("", "nan"))
        ranks = all(new[k]["rank"] == old[k]["rank"] for k in old)
        print(f"{'':8s} ranking: max |ΔW1| = {diff:.2e}, same ranks: {ranks}")